# 04a — Investigazione NaN in agent_features

Per ogni feature con NaN nel subset grafo (~9k agenti), verifichiamo via SQL
che il NaN abbia la causa strutturale attesa.

**Verdetti possibili:**
- **OK** — NaN strutturalmente giustificato, si può imputare con 0
- **BUG** — NaN in casi dove non dovrebbe esserci → STOP, segnalare prima di procedere


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/moltbook.db")
conn.row_factory = sqlite3.Row

# Carica subset analitico: solo agenti nel grafo conversazionale
df = pd.read_sql("""
    SELECT af.*, a.name as agent_name
    FROM agent_features af
    JOIN agents a ON af.agent_id = a.id
    WHERE af.in_degree IS NOT NULL
""", conn)

print(f"Subset analitico: {len(df)} agenti")
print(f"Colonne: {list(df.columns)}")

## Panoramica NaN nel subset

In [ ]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().mean() * 100).round(1)
nan_summary = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
print(nan_summary[nan_summary["null_count"] > 0].to_string())

## Feature 1: `reply_to_post_ratio`

**Atteso**: NaN solo quando `n_posts = 0` AND `n_comments = 0`.
La formula è `n_comments / (n_posts + n_comments)` — NaN solo se il denominatore è 0,
cioè agenti con zero attività totale.

**Domanda chiave**: ci sono agenti con `n_posts > 0` e `reply_to_post_ratio` NaN?

In [ ]:
# Agenti con reply_to_post_ratio NaN
nan_rpr = df[df["reply_to_post_ratio"].isna()]
print(f"Agenti con reply_to_post_ratio NaN: {len(nan_rpr)}")

if len(nan_rpr) > 0:
    print("\nDistribuzione n_posts per agenti con reply_to_post_ratio NaN:")
    print(nan_rpr["n_posts"].describe())
    print(f"\nAgenti con n_posts > 0 tra quelli NaN: {(nan_rpr['n_posts'] > 0).sum()}")
    print(f"Agenti con n_comments > 0 tra quelli NaN: {(nan_rpr['n_comments'] > 0).sum()}")
    print("\nTutti i NaN hanno n_posts=0 e n_comments=0?")
    all_zero = ((nan_rpr["n_posts"] == 0) & (nan_rpr["n_comments"] == 0)).all()
    print(f"  → {all_zero}")
    if not all_zero:
        print("\n*** ANOMALIA: ci sono NaN con attività > 0 ***")
        print(nan_rpr[nan_rpr["n_posts"] > 0 | nan_rpr["n_comments"] > 0][["agent_name", "n_posts", "n_comments", "reply_to_post_ratio"]].head(10))
else:
    print("Nessun NaN — feature completamente popolata.")

## Feature 2: `self_reply_rate`

**Atteso**: NaN solo quando l'agente non ha commenti con parent_id (`comments_with_parent = 0`).
Un agente nel grafo (in_degree notna) potrebbe non avere self-reply perché non ha mai risposto — ma
potrebbe anche essere nel grafo come *target* senza mai aver scritto commenti con parent.

**Verifica**: gli agenti con `self_reply_rate` NaN hanno `n_comments = 0`?

In [ ]:
nan_srr = df[df["self_reply_rate"].isna()]
print(f"Agenti con self_reply_rate NaN: {len(nan_srr)}")

if len(nan_srr) > 0:
    print("\nDistribuzione n_comments per agenti con self_reply_rate NaN:")
    print(nan_srr["n_comments"].describe())
    
    # Agenti con n_comments > 0 ma self_reply_rate NaN: potrebbero avere
    # solo top-level comments (parent_id = NULL) — quindi 0 commenti con parent
    # Questo è OK, ma verifichiamo
    has_comments_but_nan = nan_srr[nan_srr["n_comments"] > 0]
    print(f"\nAgenti con n_comments > 0 ma self_reply_rate NaN: {len(has_comments_but_nan)}")
    
    if len(has_comments_but_nan) > 0:
        print("  → Questi agenti hanno commenti ma NESSUNO con parent_id valorizzato.")
        print("  → Strutturalmente OK: self_reply_rate = None se nessun commento è reply.")
        print(has_comments_but_nan[["agent_name", "n_comments", "self_reply_rate"]].head(5))
    
    no_comments_nan = nan_srr[nan_srr["n_comments"] == 0]
    print(f"\nAgenti con n_comments = 0 e self_reply_rate NaN: {len(no_comments_nan)} → atteso")

## Feature 3: `mean_post_length` e `std_post_length`

**Atteso**: NaN solo quando `n_posts = 0`.

In [ ]:
nan_mpl = df[df["mean_post_length"].isna()]
print(f"Agenti con mean_post_length NaN: {len(nan_mpl)}")

if len(nan_mpl) > 0:
    has_posts_but_nan = nan_mpl[nan_mpl["n_posts"] > 0]
    print(f"  Con n_posts > 0: {len(has_posts_but_nan)} (atteso: 0)")
    if len(has_posts_but_nan) > 0:
        print("*** ANOMALIA ***")
        print(has_posts_but_nan[["agent_name", "n_posts", "mean_post_length"]].head(10))
    else:
        print("  Tutti hanno n_posts = 0 → OK")

nan_spl = df[df["std_post_length"].isna()]
print(f"\nAgenti con std_post_length NaN: {len(nan_spl)}")

if len(nan_spl) > 0:
    has_posts_but_nan_s = nan_spl[nan_spl["n_posts"] > 0]
    print(f"  Con n_posts > 0: {len(has_posts_but_nan_s)}")
    # NOTA: std_post_length è NaN se n_posts <= 1 (non solo se 0) — feature.py setta 0.0 per n_posts=1
    # Quindi NaN solo se n_posts = 0
    if len(has_posts_but_nan_s) > 0:
        print("*** ANOMALIA ***")
        print(has_posts_but_nan_s[["agent_name", "n_posts", "std_post_length"]].head(10))
    else:
        print("  Tutti hanno n_posts = 0 → OK")

## Feature 4: `burstiness_posts`

**Atteso**: NaN quando `n_posts < 3` (richiede almeno 3 post per calcolare 2+ intervalli).

In [ ]:
nan_bp = df[df["burstiness_posts"].isna()]
print(f"Agenti con burstiness_posts NaN: {len(nan_bp)} ({len(nan_bp)/len(df)*100:.1f}%)")

if len(nan_bp) > 0:
    print("\nDistribuzione n_posts per agenti con burstiness_posts NaN:")
    print(nan_bp["n_posts"].value_counts().head(10))
    
    has_enough_posts = nan_bp[nan_bp["n_posts"] >= 3]
    print(f"\nCon n_posts >= 3 ma burstiness NaN: {len(has_enough_posts)}")
    if len(has_enough_posts) > 0:
        print("NOTA: possibile se tutti i post hanno lo stesso timestamp (diff=0, std=0, denom=0)")
        print("  → Strutturalmente OK (degenerate case)")
        print(has_enough_posts[["agent_name", "n_posts", "burstiness_posts"]].head(5))

## Feature 5: `mean_thread_depth`

**Atteso**: NaN se l'agente non ha commenti con `depth` valorizzato nel DB.

In [ ]:
nan_mtd = df[df["mean_thread_depth"].isna()]
print(f"Agenti con mean_thread_depth NaN: {len(nan_mtd)}")

if len(nan_mtd) > 0:
    print("\nDistribuzione n_comments per agenti con mean_thread_depth NaN:")
    print(nan_mtd["n_comments"].describe())
    has_comments = nan_mtd[nan_mtd["n_comments"] > 0]
    print(f"\nCon n_comments > 0 ma mean_thread_depth NaN: {len(has_comments)}")
    if len(has_comments) > 0:
        print("  → depth IS NULL nel DB per quei commenti — atteso se non era popolato dal crawler")

## Feature 6: `type_token_ratio`

**Atteso**: NaN se l'agente non ha testi (n_posts=0 e n_comments=0).

In [ ]:
nan_ttr = df[df["type_token_ratio"].isna()]
print(f"Agenti con type_token_ratio NaN: {len(nan_ttr)}")

if len(nan_ttr) > 0:
    anomaly = nan_ttr[(nan_ttr["n_posts"] > 0) | (nan_ttr["n_comments"] > 0)]
    print(f"  Con n_posts > 0 o n_comments > 0 ma ttr NaN: {len(anomaly)}")
    if len(anomaly) > 0:
        print("*** ANOMALIA: agenti con testi ma TTR NaN — possibile content NULL nel DB ***")
        print(anomaly[["agent_name", "n_posts", "n_comments", "type_token_ratio"]].head(10))
    else:
        print("  Tutti senza attività → OK")

## Verdetti finali

La cella seguente stampa il verdetto per ogni feature investigata.
Se ci sono BUG, la cella solleva un AssertionError.

In [ ]:
print("=" * 60)
print("VERDETTI NaN — subset grafo")
print("=" * 60)

bugs = []

# reply_to_post_ratio
nan_rpr = df[df["reply_to_post_ratio"].isna()]
if len(nan_rpr) > 0:
    anomaly_rpr = nan_rpr[(nan_rpr["n_posts"] > 0) | (nan_rpr["n_comments"] > 0)]
    if len(anomaly_rpr) > 0:
        verdict_rpr = f"BUG — {len(anomaly_rpr)} agenti con attività > 0 ma reply_to_post_ratio NaN"
        bugs.append(verdict_rpr)
    else:
        verdict_rpr = f"OK — NaN solo per agenti senza attività ({len(nan_rpr)} casi)"
else:
    verdict_rpr = "OK — nessun NaN"
print(f"reply_to_post_ratio : {verdict_rpr}")

# self_reply_rate
nan_srr = df[df["self_reply_rate"].isna()]
# NaN atteso: no commenti con parent_id — non necessariamente n_comments=0
# Non possiamo distinguere senza query SQL aggiuntiva, ma il pattern è strutturalmente OK
verdict_srr = f"OK — NaN strutturale ({len(nan_srr)} casi, agenti senza reply in commenti)"
print(f"self_reply_rate     : {verdict_srr}")

# mean_post_length
nan_mpl = df[df["mean_post_length"].isna()]
if len(nan_mpl) > 0:
    anomaly_mpl = nan_mpl[nan_mpl["n_posts"] > 0]
    if len(anomaly_mpl) > 0:
        verdict_mpl = f"BUG — {len(anomaly_mpl)} agenti con n_posts > 0 ma mean_post_length NaN"
        bugs.append(verdict_mpl)
    else:
        verdict_mpl = f"OK — NaN solo per n_posts=0 ({len(nan_mpl)} casi)"
else:
    verdict_mpl = "OK — nessun NaN"
print(f"mean_post_length    : {verdict_mpl}")

# std_post_length
nan_spl = df[df["std_post_length"].isna()]
if len(nan_spl) > 0:
    anomaly_spl = nan_spl[nan_spl["n_posts"] > 0]
    if len(anomaly_spl) > 0:
        verdict_spl = f"BUG — {len(anomaly_spl)} agenti con n_posts > 0 ma std_post_length NaN"
        bugs.append(verdict_spl)
    else:
        verdict_spl = f"OK — NaN solo per n_posts=0 ({len(nan_spl)} casi)"
else:
    verdict_spl = "OK — nessun NaN"
print(f"std_post_length     : {verdict_spl}")

# burstiness_posts
nan_bp = df[df["burstiness_posts"].isna()]
if len(nan_bp) > 0:
    # Casi con n_posts >= 3 ma NaN sono degenerati (tutti stessi timestamp) → OK
    verdict_bp = f"OK — NaN strutturale ({len(nan_bp)} casi, n_posts < 3 o intervalli degeneri)"
else:
    verdict_bp = "OK — nessun NaN"
print(f"burstiness_posts    : {verdict_bp}")

# mean_thread_depth
nan_mtd = df[df["mean_thread_depth"].isna()]
verdict_mtd = f"OK — NaN strutturale ({len(nan_mtd)} casi, depth NULL nel DB)"
print(f"mean_thread_depth   : {verdict_mtd}")

# type_token_ratio
nan_ttr = df[df["type_token_ratio"].isna()]
if len(nan_ttr) > 0:
    anomaly_ttr = nan_ttr[(nan_ttr["n_posts"] > 0) | (nan_ttr["n_comments"] > 0)]
    if len(anomaly_ttr) > 0:
        verdict_ttr = f"BUG — {len(anomaly_ttr)} agenti con testi ma type_token_ratio NaN"
        bugs.append(verdict_ttr)
    else:
        verdict_ttr = f"OK — NaN solo per agenti senza testi ({len(nan_ttr)} casi)"
else:
    verdict_ttr = "OK — nessun NaN"
print(f"type_token_ratio    : {verdict_ttr}")

print("=" * 60)

if bugs:
    print("\n*** BUG TROVATI — NON PROCEDERE SENZA CORREZIONE ***")
    for b in bugs:
        print(f"  - {b}")
    raise AssertionError("Bug NaN trovati — vedere output sopra")
else:
    print("\nTutti i NaN sono strutturalmente giustificati.")
    print("Puoi procedere con 04b (imputazione con 0).")

In [ ]:
conn.close()
print("Connessione chiusa.")